# Agent-Centered Trajectory Metrics · Agent-Type Breakdown

Computes per-agent-type distributions for velocity, acceleration, observation duration, ego-agent spacing, path efficiency, heading change, and collision safety. The notebook loads nuScenes via trajdata in **agent-centric** mode and exports per-type statistics.


## 1. Configure dataset root
Update `NUSCENES_ROOT` if your dataset lives elsewhere.


In [ ]:
from pathlib import Path
import os

NUSCENES_ROOT = Path('../data/raw').resolve()
if not NUSCENES_ROOT.exists():
    raise FileNotFoundError(f"nuScenes root not found at {NUSCENES_ROOT}. Update the path before continuing.")

os.environ['NUSCENES_ROOT'] = str(NUSCENES_ROOT)
print(f"nuScenes root set to: {NUSCENES_ROOT}")


## 2. Imports and agent-type helpers


In [ ]:
import math
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import seaborn as sns
import matplotlib.pyplot as plt

from trajdata import UnifiedDataset
from trajdata.data_structures import AgentType
from trajdata.data_structures import agent_collate_fn

sns.set_style("whitegrid")

AGENT_TYPES = [
    AgentType.VEHICLE,
    AgentType.PEDESTRIAN,
    AgentType.BICYCLE,
    AgentType.MOTORCYCLE,
]
AGENT_TYPE_NAMES = {t.value: t.name.capitalize() for t in AGENT_TYPES}


def agent_type_name(tensor_val) -> str:
    try:
        at = AgentType(int(tensor_val))
        return AGENT_TYPE_NAMES.get(at.value, at.name.capitalize())
    except Exception:
        return str(int(tensor_val))


## 3. Dataset loader (agent-centric)
Uses trajdata's `UnifiedDataset` to stream batches of agent trajectories. The configuration mirrors the setup notebook with 3.2 s history and 4.8 s future.


In [ ]:
def load_agent_dataset(batch_size: int = 64) -> DataLoader:
    dataset = UnifiedDataset(
        desired_data=["nusc_mini"],
        centric="agent",
        desired_dt=0.1,
        history_sec=(3.2, 3.2),
        future_sec=(4.8, 4.8),
        incl_raster_map=False,
        incl_vector_map=False,
        state_format="x,y,xd,yd,xdd,ydd,h",
        obs_format="x,y,xd,yd,xdd,ydd,s,c",
        standardize_data=False,
        standardize_derivatives=False,
        rebuild_cache=False,
        num_workers=0,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=agent_collate_fn,
    )


agent_loader = load_agent_dataset()
first_batch = next(iter(agent_loader))
print("Loaded dataset with batch size", first_batch.agent_hist.shape[0])


## 4. Metric helpers
Each helper extracts per-sample values and pairs them with the corresponding agent type. Metrics cover velocity, acceleration, observation duration, ego-agent distance, path efficiency, heading change, and collision counts.


In [ ]:
def append_metric(rows, metric_name: str, types, values):
    values = values.detach().cpu().numpy() if isinstance(values, torch.Tensor) else np.asarray(values)
    for agent_type_val, metric_val in zip(types, values):
        rows.append({
            "metric": metric_name,
            "agent_type": agent_type_name(agent_type_val),
            "value": float(metric_val),
        })


def compute_agent_metric_rows(loader: DataLoader):
    rows = []
    for batch in loader:
        batch_size = batch.agent_hist.shape[0]
        dt = batch.dt.view(-1)
        agent_types = batch.agent_type

        # Velocity and acceleration from observed history
        speeds = batch.agent_hist.velocity.norm(dim=-1)
        accels = batch.agent_hist.acceleration.norm(dim=-1)
        # Use last observed timestep per agent
        last_hist_idx = (batch.agent_hist_len.clamp(min=1) - 1).clamp(max=batch.agent_hist.shape[1] - 1)
        agent_indices = torch.arange(batch_size, device=batch.agent_hist.device)
        last_speed = speeds[agent_indices, last_hist_idx]
        last_accel = accels[agent_indices, last_hist_idx]
        append_metric(rows, "velocity_mps", agent_types, last_speed)
        append_metric(rows, "acceleration_mps2", agent_types, last_accel)

        # Observation duration
        obs_duration = batch.agent_hist_len.float() * dt
        append_metric(rows, "observation_duration_s", agent_types, obs_duration)

        # Path efficiency: straight-line displacement vs. traveled distance
        full_positions = torch.cat([batch.agent_hist.position, batch.agent_fut.position], dim=1)
        total_len = batch.agent_hist_len + batch.agent_fut_len
        max_len = full_positions.shape[1]
        step_mask = torch.arange(max_len - 1, device=full_positions.device).unsqueeze(0) < (total_len - 1).unsqueeze(1)
        path_deltas = full_positions[:, 1:] - full_positions[:, :-1]
        step_lengths = path_deltas.norm(dim=-1) * step_mask
        path_length = step_lengths.sum(dim=1)
        last_idx = (total_len - 1).clamp(min=0, max=max_len - 1)
        last_pos = full_positions[agent_indices, last_idx]
        displacement = (last_pos - full_positions[:, 0]).norm(dim=-1)
        path_efficiency = torch.where(path_length > 0, displacement / path_length, torch.nan)
        append_metric(rows, "path_efficiency", agent_types, path_efficiency)

        # Heading change across the full trajectory
        full_headings = torch.cat([batch.agent_hist.heading, batch.agent_fut.heading], dim=1)
        heading_change = torch.abs(full_headings[:, -1] - full_headings[:, 0])
        append_metric(rows, "heading_change_rad", agent_types, heading_change)

        # Ego-agent distance: nearest neighbor at last observed timestep
        if batch.neigh_hist.numel() > 0:
            neigh_positions = batch.neigh_hist.position
            neigh_len = batch.neigh_hist_len
            neigh_last_idx = (neigh_len.clamp(min=1) - 1).clamp(max=neigh_positions.shape[2] - 1)
            gather_idx = neigh_last_idx.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, 1, neigh_positions.shape[-1])
            last_neigh_pos = torch.gather(neigh_positions, 2, gather_idx).squeeze(2)
            last_agent_pos = batch.agent_hist.position[agent_indices, last_hist_idx].unsqueeze(1)
            valid_neigh = neigh_len > 0
            dists = torch.norm(last_agent_pos - last_neigh_pos, dim=-1)
            dists = torch.where(valid_neigh, dists, torch.full_like(dists, float("inf")))
            min_dist, _ = dists.min(dim=1)
            append_metric(rows, "ego_agent_min_distance_m", agent_types, min_dist)
        else:
            append_metric(rows, "ego_agent_min_distance_m", agent_types, torch.full((batch_size,), float("nan")))

        # Safety: count neighbors within 2 m at last observed timestep
        if batch.neigh_hist.numel() > 0:
            collision_mask = (dists < 2.0) & valid_neigh
            near_collisions = collision_mask.sum(dim=1).float()
            append_metric(rows, "collisions_within_2m", agent_types, near_collisions)
        else:
            append_metric(rows, "collisions_within_2m", agent_types, torch.zeros(batch_size))

    return pd.DataFrame(rows)


## 5. Run metric computation and export per-type statistics


In [ ]:
agent_metrics_df = compute_agent_metric_rows(agent_loader)
agent_metrics_df.head()

summary = (
    agent_metrics_df
    .groupby(["metric", "agent_type"])["value"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)

metrics_dir = Path("../results/metrics")
metrics_dir.mkdir(parents=True, exist_ok=True)
agent_metrics_df.to_csv(metrics_dir / "agent_centered_metrics_by_type.csv", index=False)
summary.to_csv(metrics_dir / "agent_centered_metrics_by_type_summary.csv", index=False)
summary.head()


## 6. Visualize distributions by agent type
Grouped and faceted plots for the per-type comparisons.


In [ ]:
plot_order = ["Vehicle", "Pedestrian", "Bicycle", "Motorcycle"]
def plot_distribution(metric_name, bins=40):
    subset = agent_metrics_df[agent_metrics_df.metric == metric_name]
    g = sns.displot(
        subset,
        x="value",
        hue="agent_type",
        hue_order=plot_order,
        kind="hist",
        bins=bins,
        facet_kws={"sharex": True, "sharey": False},
    )
    g.fig.suptitle(f"{metric_name} by agent type", y=1.02)
    return g

plot_distribution("velocity_mps")
plot_distribution("acceleration_mps2")
plot_distribution("observation_duration_s", bins=20)

bar_data = summary[summary.metric.isin([
    "path_efficiency", "heading_change_rad", "ego_agent_min_distance_m", "collisions_within_2m"
])]
plt.figure(figsize=(10, 4))
sns.barplot(data=bar_data, x="metric", y="mean", hue="agent_type", hue_order=plot_order)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Mean value")
plt.title("Agent-type comparisons (means)")
plt.tight_layout()
plt.show()
